<a href="https://colab.research.google.com/github/imaniiz/CariSurg-Portfolio/blob/feat%2Fweek-6-refactor/Notebooks/Week6_Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CARISURG MedTech Pathways | Heathcare AI Programme | Week 6 Assignment
#**AI-Assisted Triage: Baseline Model Development**

---

##**About this Notebook**

This notebook builds upon the exploratory data anlysis and data quality assessment completed in Week 5 by delivered developing the first baseline machine learning model for AI-assisted emergency department triage. Using the cleaned Yale EMMLC dataset, the objective is to determine whether simple, interpretable models can predict Emergency Severity Index (ESI) levels better than a random baseline.

##Baseline Classification Algorithms:
1. Logistic Regression
2. Decision Tree Classification  

The performance of the algorithms is evaluated using metrics such as accuracy, precision, recall and F1-score and compared against a stratified random classifier.

##Research Question:
Can simple, interpretable machine learning models accurately predict Emergenyc Severity Index levels and which evaluation metricbest relflects clinically safe triage performance?

##Notebook Sections

Section 1: Data Loading & Environment Setup

Section 2: Feature Selection & Target Variable

Section 3: Training/Test Split

Section 4: Baseline Random Classifier

Section 5: Model 1 - Logistic Regression

Section 6: Model 2 - Decision Tree  

Section 7: Model Evaluation
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

Section 8: Model Comparison

Section 9: Clinical Metric Justification

Section 10: Failure Modes and Clinical Implications

Section 11: Confusion



##**Section 1: Data Loading & Environment Setup**

In [2]:
# Importing relevant python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

pd.set_option("display.width", 120)
print("Libraries loaded. ✅")

Libraries loaded. ✅


In [3]:
# Environment setup
# Reloading cleaned triage dataset from week 5
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

CLEAN_PATH = Path('/content/drive/MyDrive/Carisurg Portfolio/CariSurg_Week5/triage_cleaned_v1.csv')

df = pd.read_csv(CLEAN_PATH)

# Printing the shape and first five rows of the dataset
print(df.shape)
print("Loaded", df.shape[0], "patients and", df.shape[1], "columns.")
df.head()

Mounted at /content/drive
(55121, 225)
Loaded 55121 patients and 225 columns.


,dep_name,esi,age,gender,ethnicity,race,lang,religion,maritalstatus,employstatus,...,cc_vaginaldischarge,cc_vaginalpain,cc_weakness,cc_wheezing,cc_withdrawal-alcohol,cc_woundcheck,cc_woundinfection,cc_woundre-evaluation,cc_wristinjury,cc_wristpain
0,A,4,87.0,Female,Hispanic or Latino,Other,Other,Pentecostal,Widowed,Retired,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,B,2,53.0,Male,Hispanic or Latino,Other,English,Catholic,Significant Other,Disabled,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,A,2,49.0,Female,Non-Hispanic,White or Caucasian,English,Catholic,Married,Full Time,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A,3,22.0,Female,Hispanic or Latino,White or Caucasian,English,Catholic,Single,Full Time,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,A,2,62.0,Male,Non-Hispanic,White or Caucasian,English,Protestant,Divorced,Not Employed,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##Section 2: Feature Selection & Target Variable

In [4]:
TARGET = "esi"

# Vital-sign columns measured at the front door:
VITALS = ["triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
          "triage_vital_o2", "triage_vital_temp", "triage_glucose"]
# Who the patient is (some of these are fairness-sensitive — handle with care):
DEMOGRAPHICS = ["age", "gender", "ethnicity", "race", "lang", "religion",
                "maritalstatus", "employstatus", "insurance_status"]
# Administrative / arrival details:
ADMIN = ["dep_name", "arrivalmode", "arrivalmonth", "arrivalday", "arrivalhour_bin"]
# OUTCOMES of the visit — known only AFTER triage, so they must never be model inputs:
LEAKAGE = ["disposition", "previousdispo"]

FEATURES = [c for c in df.columns if c != TARGET and c not in LEAKAGE + ADMIN + DEMOGRAPHICS]

In [5]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: decides which columns the model is allowed to see.
#
#   TARGET  = the thing we want to PREDICT  -> esi (the triage level 1-5)
#   FEATURES = the clues the model may use to make that prediction
#
# WHY we DROP "disposition": it records what happened AFTER triage
# (admitted/discharged). Letting the model peek at the future is called
# "data leakage" — it makes scores look great but the model is useless
# in real life. So we remove it (and any other after-the-fact columns).
# ------------------------------------------------------------------

X = df[FEATURES]     # the clues  (a table: one row per patient)
y = df[TARGET]       # the answer (one ESI level per patient)

print("Model will use", len(FEATURES), "features to predict:", TARGET)
print("First few features:", FEATURES[:6], "...")

Model will use 208 features to predict: esi
First few features: ['triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr', 'triage_vital_o2', 'triage_vital_o2_device'] ...


##Section 3: Training/Test Split


In [6]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: separates a TEST set (20%) that the model will not
# see during training, so later we can grade it honestly.
# ------------------------------------------------------------------
# TODO — fill in train_test_split(...)
#   - the function is train_test_split(X, y, ...)
#   - test_size= keeps 20% for testing
#   - stratify=y keeps the ESI mix balanced across both sets
#   - random_state= (any fixed number) makes the split reproducible

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Training patients:", X_train.shape[0])
print("Testing patients: ", X_test.shape[0])

Training patients: 44096
Testing patients:  11025


In [7]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: builds the "floor" — a model that only guesses.
# .fit() = learn from training data.  .score() = fraction correct on test.
# ------------------------------------------------------------------
# TODO —
#   - make a DummyClassifier(strategy="stratified", random_state=42)
#   - train it with .fit(X_train, y_train)
#   - print its .score(X_test, y_test)
# dummy = ...

dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)
print("Dummy classifier accuracy:"), dummy.score(X_test, y_test)


Dummy classifier accuracy:


(None, 0.37541950113378686)

Excersize question 1:

Add clinical question: Re-run the split with a different random_state. Do the accuracies move a little? Why?

In [ ]:
# Re-run split here

###Observations

##Section 4: Baseline Random Classifier

##Section 5: Model 1 - Logistic Regression

In [ ]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: (1) rescales features so they're comparable, then
# (2) trains logistic regression on the scaled training data.
# ------------------------------------------------------------------
# TODO part A — scale the features:
#   - make a StandardScaler()
#   - use scaler.fit_transform(X_train) for the training features
#   - use scaler.transform(X_test) for the test features (same scaling!)
# TODO part B — train the model:
#   - make a LogisticRegression(max_iter=1000, random_state=42)
#   - .fit(...) it on the SCALED training features and y_train
#   - print its .score(...) on the SCALED test features and y_test


###Add Clinical question for logistic regression plot

In [ ]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: shows the "dividing line" idea on YOUR data.
# We can't draw 20 features at once, so we pick just TWO vitals and ask a
# simple TWO-class question ("urgent?" = ESI 1 or 2), then let logistic
# regression draw the line that best separates the two groups.
# ------------------------------------------------------------------
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1) pick two features for the x and y axes (try swapping these!)
f1, f2 = None, None
# f1, f2 = "triage_vital_o2", "triage_vital_rr"
# f1, f2 = "triage_vital_temp", "triage_vital_hr"
# NOTE: try your own combinations!

# 2) make a simple two-class label: urgent (ESI 1-2) vs not urgent (ESI 3-5)
urgent = (y <= 2).astype(int)          # 1 = urgent, 0 = not urgent

# 3) scale the two features (logistic regression likes balanced scales — see section 4)
two = StandardScaler().fit_transform(X[[f1, f2]])

# 4) train a small logistic regression on just those two features
demo = LogisticRegression(max_iter=1000).fit(two, urgent)

# 5) cover the plot with a grid of points and ask the model to classify each,
#    so we can shade the two "decision regions"
xx, yy = np.meshgrid(np.linspace(two[:, 0].min(), two[:, 0].max(), 200),
                     np.linspace(two[:, 1].min(), two[:, 1].max(), 200))
zz = demo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, zz, alpha=0.2, cmap="coolwarm")                 # the two decision regions
ax.scatter(two[:, 0], two[:, 1], c=urgent, cmap="coolwarm",
           s=10, alpha=0.4, edgecolor="none")                      # the patients
ax.set_xlabel(f1 + " (standardised)")
ax.set_ylabel(f2 + " (standardised)")
ax.set_title("Logistic regression divides patients with a line (blue = not urgent, red = urgent)")
plt.tight_layout()
plt.savefig("figs/w6_logreg_boundary.png", dpi=120, bbox_inches="tight")
plt.show()

Excersize question 5:

In the logistic-regression picture, which corner of the plot is ‘urgent’? Swap f1/f2 for two other vitals — does the line separate the classes better or worse?

In [ ]:
# Do swap here

###Observations and answer question

##Section 6: Model 2 - Decision Tree

In [ ]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: trains a decision tree. No scaling needed here.
# max_depth=5 keeps the tree shallow -> easier to explain, less overfitting.
# ------------------------------------------------------------------
# TODO —
#   - make a DecisionTreeClassifier(max_depth=5, random_state=42)
#   - .fit(...) it on the UNSCALED X_train and y_train
#   - print its .score(...) on X_test and y_test
# tree = ...


###Add clinical question for decision tree plot


In [ ]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: draws YOUR trained tree as a flowchart.
# We only draw the top 2 levels so it stays readable on screen
# (your tree is actually deeper). Try changing max_depth to 1 or 3.
# ------------------------------------------------------------------
from sklearn.tree import plot_tree
import os

os.makedirs("figs", exist_ok=True)
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    tree,                                             # the tree you trained in section 5
    feature_names=FEATURES,                           # show real column names on each split
    class_names=[f"ESI {c}" for c in tree.classes_],  # label each leaf with its ESI level
    filled=True,      # colour each box by the ESI level it predicts
    rounded=True,
    max_depth=2,      # only DRAW the top 2 levels (change me to see more)
    fontsize=9,
    ax=ax,
)
ax.set_title("Your decision tree — top levels (left = yes, right = no)")
plt.savefig("figs/w6_decision_tree.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: prints the SAME tree as plain text. Handy on a
# small screen, and it shows the exact numbers the tree splits on.
# |--- means one step deeper; "class:" lines are the predicted ESI level.
# ------------------------------------------------------------------
from sklearn.tree import export_text
print(export_text(tree, feature_names=list(FEATURES), max_depth=3))

Excersize question 4: What is the first feature the tree splits on and why might that be clinically sensible for triage?

Excersize question 2

Change the tree's max_depth to 3, then to 12. What happens to training vs test accuracy — and to how readable the tree picture is?

In [ ]:
# Change tree max depth here

###Observations and answer question

Excersize question 3: why did we scale for logistic regression but not for tree?

In [ ]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: saves the models (and the scaler) to files so we
# can reuse them later without retraining.
# ------------------------------------------------------------------
joblib.dump(logreg, "model_logreg.joblib")
joblib.dump(tree,   "model_tree.joblib")
joblib.dump(scaler, "scaler.joblib")
print("Saved: model_logreg.joblib, model_tree.joblib, scaler.joblib ✅")

##Section 7: Model Evaluation

##Section 8: Model Comparison

##Section 9: Clinical Metric Justification

##Section 10: Failure Modes & Clinical Implications

##Section 11: Conclusion